# Visualize retrieval datasets and pipeline results

This notebook reads the append-only dataset and result registries. Dataset statistics and evaluation metrics are restricted to the latest registered version of each dataset name.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/retrieval-benchlab")
if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        !git clone --depth 1 https://github.com/lohex/retrieval-benchlab.git {REPO_ROOT}
    else:
        !git -C {REPO_ROOT} pull --ff-only
    %cd {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

!pip -q install -U datasets pandas matplotlib seaborn jinja2


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.io import mount_google_drive
from src.reporting import load_registry_report

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)


## Configuration

Set `METRICS_TO_PLOT` to a list such as `["ndcg@10", "map@100"]` to restrict the figures. The default `None` plots every stored metric.

In [ ]:
REGISTRY_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/datasets.sqlite"
RESULTS_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/results.sqlite"
METRICS_TO_PLOT = None


In [ ]:
mount_google_drive()
report = load_registry_report(
    registry_db_path=REGISTRY_DB_PATH,
    results_db_path=RESULTS_DB_PATH,
)


## Latest dataset versions

`unique_positive_documents` counts distinct positive documents in a dataset. `positive_relations` counts query-document relevance pairs.

In [ ]:
dataset_columns = [
    "dataset", "version", "documents", "queries",
    "unique_positive_documents", "negative_documents",
    "positive_relations", "positive_relations_per_query",
    "unique_positives_per_query",
]
dataset_table = report.datasets[dataset_columns]
display(dataset_table.style.format(
    {
        "documents": "{:,.0f}",
        "queries": "{:,.0f}",
        "unique_positive_documents": "{:,.0f}",
        "negative_documents": "{:,.0f}",
        "positive_relations": "{:,.0f}",
        "positive_relations_per_query": "{:.2f}",
        "unique_positives_per_query": "{:.2f}",
    },
    na_rep="n/a",
).hide(axis="index"))


In [ ]:
dataset_plot_specs = [
    ("documents", "Corpus documents", "%.0f"),
    ("queries", "Queries", "%.0f"),
    ("unique_positive_documents", "Unique positive documents", "%.0f"),
    ("positive_relations_per_query", "Mean relevant documents per query", "%.2f"),
]
figure, axes = plt.subplots(2, 2, figsize=(14, 9))
for axis, (column, title, label_format) in zip(axes.flat, dataset_plot_specs, strict=True):
    sns.barplot(data=report.datasets, y="dataset", x=column, ax=axis)
    axis.set_title(title)
    axis.set_xlabel("")
    axis.set_ylabel("")
    axis.bar_label(axis.containers[0], fmt=label_format, padding=3)
    axis.margins(x=0.15)
figure.suptitle("Latest registered BioASQ datasets", fontsize=16)
figure.tight_layout()
plt.show()


## Pipeline coverage

Coverage is the fraction of latest dataset versions for which a pipeline has a stored result.

In [ ]:
coverage_columns = [
    "pipeline_label", "model_name", "similarity_metric",
    "evaluated_latest_datasets", "available_latest_datasets", "coverage",
]
coverage_table = report.pipelines[coverage_columns].sort_values(
    ["coverage", "pipeline_label"], ascending=[False, True]
)
display(coverage_table.style.format({"coverage": "{:.0%}"}).hide(axis="index"))


## Pipeline comparison by metric

Bars show the unweighted mean across latest datasets and points show individual dataset values.

In [ ]:
def ordered_pipeline_table(pipelines):
    ordered = pipelines.sort_values(
        ["model_name", "similarity_metric", "pipeline_id"]
    ).copy()
    model_short_name = ordered["model_name"].str.rsplit("/", n=1).str[-1]
    ordered["plot_label"] = (
        model_short_name + "\n" + ordered["similarity_metric"]
        + "\n" + ordered["pipeline_id"].str[-6:]
    )
    return ordered


def summarize_metric(metric_values, pipelines):
    summary = (
        metric_values.groupby("pipeline_id")
        .agg(mean_value=("value", "mean"), evaluated_datasets=("dataset", "nunique"))
        .reset_index()
    )
    return pipelines.merge(summary, how="left", on="pipeline_id", validate="one_to_one")


def plot_metric(metric_name, report, pipelines):
    metric_values = report.metrics[report.metrics["metric"] == metric_name].copy()
    summary = summarize_metric(metric_values, pipelines)
    positions = list(range(len(summary)))
    position_by_pipeline = dict(zip(summary["pipeline_id"], positions, strict=True))

    figure_width = max(10.0, 2.8 * len(summary))
    figure, axis = plt.subplots(figsize=(figure_width, 7))
    bars = axis.bar(positions, summary["mean_value"].fillna(0.0))
    available_datasets = report.datasets["dataset_id"].nunique()

    for position, bar, mean_value, dataset_count in zip(
        positions, bars, summary["mean_value"], summary["evaluated_datasets"], strict=True
    ):
        if pd.isna(mean_value):
            bar.set_facecolor("none")
            axis.text(position, 0.02, "no result", ha="center", va="bottom")
            continue
        axis.text(
            position, float(mean_value) + 0.015,
            f"{mean_value:.3f}\n{int(dataset_count)}/{available_datasets} datasets",
            ha="center", va="bottom",
        )

    dataset_names = sorted(metric_values["dataset"].unique())
    offsets = [
        0.0 if len(dataset_names) == 1 else -0.18 + 0.36 * index / (len(dataset_names) - 1)
        for index in range(len(dataset_names))
    ]
    for dataset_name, offset in zip(dataset_names, offsets, strict=True):
        dataset_values = metric_values[metric_values["dataset"] == dataset_name]
        point_positions = [
            position_by_pipeline[pipeline_id] + offset
            for pipeline_id in dataset_values["pipeline_id"]
        ]
        axis.scatter(point_positions, dataset_values["value"], label=dataset_name, s=42, zorder=3)

    axis.set_xticks(positions, labels=summary["plot_label"])
    axis.set_title(f"{metric_name}: all registered pipelines")
    axis.set_xlabel("Pipeline")
    axis.set_ylabel(metric_name)
    if metric_values["value"].between(0.0, 1.0).all():
        axis.set_ylim(0.0, 1.08)
    axis.legend(title="Dataset", bbox_to_anchor=(1.02, 1.0), loc="upper left")
    figure.tight_layout()
    plt.show()


available_metrics = sorted(report.metrics["metric"].unique())
metric_names = available_metrics
if METRICS_TO_PLOT is not None:
    unknown_metrics = set(METRICS_TO_PLOT).difference(available_metrics)
    if unknown_metrics:
        raise ValueError(f"Unknown metrics: {sorted(unknown_metrics)}")
    metric_names = list(METRICS_TO_PLOT)

if not metric_names:
    print("No metrics are stored for the latest dataset versions.")
else:
    pipelines = ordered_pipeline_table(report.pipelines)
    for metric_name in metric_names:
        plot_metric(metric_name, report, pipelines)


## Optional database reset

Set `RESET_DATABASES = True` and run the following cell to delete both SQLite databases. They will be recreated with the current schema by the next dataset-registration or evaluation run.

In [ ]:
RESET_DATABASES = False

if RESET_DATABASES:
    for database_path in (Path(REGISTRY_DB_PATH), Path(RESULTS_DB_PATH)):
        for suffix in ("", "-wal", "-shm"):
            path = Path(f"{database_path}{suffix}")
            if path.exists():
                path.unlink()
                print(f"Deleted {path}")
else:
    print("Database reset disabled. Set RESET_DATABASES = True to enable it.")
